# Optimizer profile: scaling with model size
Step time and peak memory against parameter count $P$, Sven against the baselines. Two families: the 3-hidden-layer MNIST MLP (width 32 to 2048) and the 4-layer nanoGPT (embedding width 64 to 512). Red crosses mark the first size at which a method ran out of memory or was analytically infeasible.

In [ ]:
import sys
sys.path.insert(0, '.')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from style import set_style
import profile_helpers as ph
set_style()
PLOT_DIR = 'plots_v2/profile'
df = ph.load_profiles()          # ../profile_results_v2/*/*.json -> one tidy row per configuration
ARCHS = [a for a in ph.ARCH_ORDER if a in set(df.arch)]
df.groupby(['arch', 'study']).size().unstack(fill_value=0)
WIDTH = [a for a in ['mnist_width', 'nanogpt_width'] if a in set(df.arch)]

In [ ]:
for value, fname in [('step_ms', 'scaling_step_time'), ('peak_mb', 'scaling_peak_memory'), ('overhead_mb', 'scaling_memory_overhead')]:
    fig, axes = plt.subplots(1, len(WIDTH), figsize=(7.5 * len(WIDTH), 5.5), squeeze=False)
    for ax, arch in zip(axes[0], WIDTH):
        ph.plot_sweep(df, arch, 'width', 'n_params', ax, value=value, analytic='analytic_jac_mb' if value != 'step_ms' else None)
        ax.set_xscale('log')
    ph.legend_below(fig, axes[0][0])
    fig.tight_layout(); ph.save(fig, f'{fname}.pdf', PLOT_DIR); plt.show()

### Sven variants only
The Gram/hooks path never forms the Jacobian, so its memory should track the model; the full-Jacobian paths carry the $B P\cdot 4$-byte block (dotted).

In [ ]:
fig, axes = plt.subplots(2, len(WIDTH), figsize=(7 * len(WIDTH), 9), squeeze=False)
for j, arch in enumerate(WIDTH):
    ms = ph.SVEN + ['Adam', 'AdamW']
    ph.plot_sweep(df, arch, 'width', 'n_params', axes[0][j], value='step_ms', methods=ms); axes[0][j].set_xscale('log')
    ph.plot_sweep(df, arch, 'width', 'n_params', axes[1][j], value='peak_mb', methods=ms, analytic='analytic_jac_mb'); axes[1][j].set_xscale('log')
ph.legend_below(fig, axes[0][0])
fig.tight_layout(); ph.save(fig, 'scaling_sven_variants.pdf', PLOT_DIR); plt.show()

### Fitted growth
Exponent of cost in $P$ over the upper half of the size range, and where each method first fails.

In [ ]:
for arch in WIDTH:
    for value in ['step_ms', 'peak_mb']:
        t = ph.scaling_table(df, arch, value)
        print(f"\n=== {ph.ARCH_TITLES[arch]}: {value} ==="); display(t)
        t.to_csv(f'{PLOT_DIR}/table_scaling_{arch}_{value}.csv', index=False)

### Relative to Adam

In [ ]:
fig, axes = plt.subplots(1, len(WIDTH), figsize=(7 * len(WIDTH), 4.8), squeeze=False)
for ax, arch in zip(axes[0], WIDTH):
    ph.plot_sweep(df, arch, 'width', 'n_params', ax, value='rel_time'); ax.set_xscale('log'); ax.set_ylabel('Step time / Adam')
ph.legend_below(fig, axes[0][0])
fig.tight_layout(); ph.save(fig, 'scaling_relative_to_adam.pdf', PLOT_DIR); plt.show()